In [1]:
"""Train a baseline gradient-boosting model per horizon and write predictions.parquet.

Baseline approach: histogram GBM regression (sklearn's LightGBM-equivalent)
on each target (target_10d, target_30d), with a time-based holdout for a
sanity-check Spearman score. Predictions are rank-normalized to [0, 1]
per the submission spec (id, pred_10d, pred_30d).
"""

from pathlib import Path

import polars as pl
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression, Lasso

DATA_DIR = Path("data") # folder
OUT_DIR = Path("predictions") # folder
OUT_DIR.mkdir(exist_ok=True)

train = pl.read_parquet(DATA_DIR / "training_data.parquet")
infer = pl.read_parquet(DATA_DIR / "inference_data.parquet")

feature_cols = [c for c in train.columns if c.startswith("feature_")] # extract the feature columns
print(f"{len(feature_cols)} features, {train.height:,} training rows, {infer.height} inference rows")

180 features, 194,852 training rows, 170 inference rows


In [2]:
def ts_split(train, lookback):
    # time-based holdout: last 60 dates for validation
    dates = train["date"].unique().sort()
    split_date = dates[-lookback] # validation holdout
    tr = train.filter(pl.col("date") < split_date) # filter up to last training_date
    va = train.filter(pl.col("date") >= split_date) # validation date and beyond
    return tr, va

In [3]:
def Ridge_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        model = Ridge(alpha=1.0, random_state=42)
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['Ridge'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]
        preds['Ridge'][f"val_{horizon}"] = pl.Series(val_pred).rank() / len(val_pred) # keep rank-normalized val preds for the ensemble check

    return preds # returns the predictions

In [4]:
def ElasticNet_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        # alpha is much smaller than Ridge's: ElasticNet's L1 part zeroes out
        # coefficients aggressively, and at alpha=1.0 it would kill all 180
        model = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42)
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['ElasticNet'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]
        preds['ElasticNet'][f"val_{horizon}"] = pl.Series(val_pred).rank() / len(val_pred) # keep rank-normalized val preds for the ensemble check

    return preds # returns the predictions

In [5]:
def Linear_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        model = LinearRegression() # plain OLS - no regularization, no knobs
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['LR'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]
        preds['LR'][f"val_{horizon}"] = pl.Series(val_pred).rank() / len(val_pred) # keep rank-normalized val preds for the ensemble check

    return preds # returns the predictions

In [ ]:
def Lasso_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        model = Lasso(alpha=0.001) # plain OLS - no regularization, no knobs
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['LASSO'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]
        preds['LASSO'][f"val_{horizon}"] = pl.Series(val_pred).rank() / len(val_pred) # keep rank-normalized val preds for the ensemble check

    return preds # returns the predictions

In [9]:
lookback = 60
MODELS = ["Ridge", "ElasticNet", "LR", "LASSO"] # one key per model
preds = {name: {"id": infer["id"]} for name in MODELS} # predictions

tr, va = ts_split(train, lookback)
print("-- Ridge --")
preds = Ridge_regression(tr, va, preds)
print("-- ElasticNet --")
preds = ElasticNet_regression(tr, va, preds)
print("-- LinearRegression --")
preds = Linear_regression(tr, va, preds)
#print("-- HGBR --")
#final = HGBR(tr, va, preds)
print("--LASSO--")
final = Lasso_regression(tr, va, preds)

-- Ridge --
target_10d: holdout Spearman = 0.0920
target_30d: holdout Spearman = 0.1078
-- ElasticNet --
target_10d: holdout Spearman = 0.1073
target_30d: holdout Spearman = 0.1671
-- LinearRegression --
target_10d: holdout Spearman = 0.0918
target_30d: holdout Spearman = 0.1074
--LASSO--
target_10d: holdout Spearman = nan


/var/folders/l2/g69cvpl114s7rx3k9xl6z7fw0000gn/T/ipykernel_5511/2933244268.py:10: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
/var/folders/l2/g69cvpl114s7rx3k9xl6z7fw0000gn/T/ipykernel_5511/2933244268.py:10: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target


target_30d: holdout Spearman = nan


-- Ridge --
target_10d: holdout Spearman = 0.0893
target_30d: holdout Spearman = 0.0989
-- ElasticNet --
target_10d: holdout Spearman = 0.0992
target_30d: holdout Spearman = 0.1562
-- LinearRegression --
target_10d: holdout Spearman = 0.0890
target_30d: holdout Spearman = 0.0986
-- HGBR --
target_10d: holdout Spearman = 0.0762
target_30d: holdout Spearman = 0.1708

In [10]:
# Convert the predictions for display/inspection (better structure)
# one column per model+horizon, joined on 'id' - ready for aggregation later
df = None
for name in MODELS:
    df_m = pl.DataFrame({
        "id": final[name]["id"],
        f"{name}_pred_10d": final[name]["pred_10d"],
        f"{name}_pred_30d": final[name]["pred_30d"],
    })
    df = df_m if df is None else df.join(df_m, on="id", how="inner")

df.head()

id,Ridge_pred_10d,Ridge_pred_30d,ElasticNet_pred_10d,ElasticNet_pred_30d,LR_pred_10d,LR_pred_30d,LASSO_pred_10d,LASSO_pred_30d
str,f64,f64,f64,f64,f64,f64,f64,f64
"""0G""",0.311765,0.270588,0.170588,0.158824,0.317647,0.270588,0.502941,0.502941
"""2Z""",0.235294,0.129412,0.088235,0.052941,0.235294,0.129412,0.502941,0.502941
"""AAVE""",0.788235,0.623529,0.682353,0.758824,0.788235,0.611765,0.502941,0.502941
"""ACE""",0.058824,0.2,0.2,0.223529,0.064706,0.205882,0.502941,0.502941
"""ADA""",0.894118,0.758824,0.805882,0.817647,0.894118,0.752941,0.502941,0.502941


In [11]:
def aggregate(final):
    # equal-weight ensemble: average the four models' rank columns per horizon,
    # then re-rank the average back to (0, 1] so the output is a valid submission
    agg = {"id": final[MODELS[0]]["id"]} # ids are identical across models
    for horizon in ["pred_10d", "pred_30d"]:
        stacked = pl.DataFrame({name: final[name][horizon] for name in MODELS}) # one column per model
        mean_rank = stacked.mean_horizontal() # average the 4 rank predictions per asset
        agg[horizon] = mean_rank.rank() / len(mean_rank) # re-rank to (0, 1]
    return pl.DataFrame(agg) # submission format: id, pred_10d, pred_30d

In [12]:
# ensemble sanity check: same equal-weight aggregation as aggregate(), but on the
# validation holdout, so we can Spearman the aggregate against the actual targets
for target in ["target_10d", "target_30d"]:
    horizon = target.replace("target", "pred")
    va_t = va.drop_nulls(subset=[target]) # same rows every model predicted on
    stacked = pl.DataFrame({name: final[name][f"val_{horizon}"] for name in MODELS}) # one column per model
    mean_rank = stacked.mean_horizontal() # average the 4 rank predictions per asset
    corr, _ = spearmanr(va_t[target].to_numpy(), mean_rank.to_numpy())
    print(f"{target}: ensemble holdout Spearman = {corr:.4f}")

target_10d: ensemble holdout Spearman = 0.0988
target_30d: ensemble holdout Spearman = 0.1293


In [13]:
submission = aggregate(final) # final targets: aggregate pred_10d and pred_30d

# sanity checks against the submission spec before writing
assert submission.columns == ["id", "pred_10d", "pred_30d"] # exact required columns
assert submission["pred_10d"].is_between(0, 1).all() # floats in [0, 1]
assert submission["pred_30d"].is_between(0, 1).all()
assert submission.height >= 80 # minimum 80 assets

submission.write_parquet(OUT_DIR / "model_1_2.parquet") # wrap it up in a parquet
print(f"wrote {OUT_DIR / 'model_1_2.parquet'} ({submission.height} assets)")
submission.head()

wrote predictions/model_1_2.parquet (170 assets)


id,pred_10d,pred_30d
str,f64,f64
"""0G""",0.276471,0.241176
"""2Z""",0.164706,0.088235
"""AAVE""",0.767647,0.682353
"""ACE""",0.105882,0.211765
"""ADA""",0.870588,0.788235
